In [ ]:
# =============================================================================
# 03_GSSINDy.py
# Group-Sparse SINDy on any registered ODESystem.
# Requires: git clone https://github.com/lindliu/GS-SINDy
# Falls back to PySINDy if the repo is absent.
# =============================================================================

# %% ── USER PARAMETERS ────────────────────────────────────────────────────────

import sys, os
sys.path.insert(0, os.getcwd())
from ode_systems import SYSTEMS, list_systems

# System selection
SYSTEM_KEY = "lorenz"
SYSTEM     = SYSTEMS[SYSTEM_KEY]

# Multiple initial conditions — GS-SINDy fits them jointly.
# Each row must have SYSTEM.n_dim entries.
X0_LIST = [
    [-8.0,  8.0, 27.0],
    [ 0.0,  1.0, 20.0],
    [ 5.0, -5.0, 10.0],
]

# Integration
T_SPAN      = (0.0, 10.0)
DT          = 0.01
NOISE_LEVEL = 0.00
SEED        = 42

# Candidate library
POLY_DEGREE     = 2
CUSTOM_BASIS_FN = None   # optional callable () → {'functions':..., 'names':...}

# GS-SINDy thresholds
# threshold_sindy      — STLSQ sparsity (same role as in plain SINDy)
# threshold_group      — zero out terms whose mean |coeff| < this
# threshold_similarity — Wasserstein distance below which a term is
#                        considered structurally identical across trajectories
THRESHOLD_SINDY      = 0.05
THRESHOLD_GROUP      = 0.01
THRESHOLD_SIMILARITY = 0.1
ALPHA                = 0.05
MAX_ITER             = 20

# Sub-series parameters
# GS-SINDy slices each trajectory into overlapping windows and fits
# coefficient distributions, which drive the group-sparsity logic.
NUM_SERIES   = 100    # windows per trajectory
WINDOW_PER   = 0.7    # fractional window length  (0 < WINDOW_PER < 1)
REMOVE_PER   = 0.2    # fraction of outlier windows to discard
DERIV_SPLINE = True   # True = spline derivative;  False = finite difference

# Evaluation
T_EVAL_END         = 5.0    # trajectory comparison window
EVAL_TRAJ          = 0      # which trajectory index to plot
SWEEP_NOISE_LEVELS = [0.0, 0.01, 0.05, 0.1, 0.2]

RESULTS_FILE = f"results/gsindy_{SYSTEM_KEY}_results.pkl"

# %% ── IMPORTS ─────────────────────────────────────────────────────────────────

import pickle, time, warnings
from itertools import combinations_with_replacement
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt
import matplotlib; matplotlib.rcParams["figure.dpi"] = 110
import pysindy as ps

from ode_utils import (
    generate_data, generate_multiple_trajectories, simulate_from_coefficients,
    plot_phase_portrait, plot_trajectories, plot_coefficient_comparison,
    plot_noise_sweep, relative_l2_error, coefficient_error, precision_recall,
    print_metrics, metrics_dict, build_true_coef_matrix,
    print_discovered_equations,
)

os.makedirs("results", exist_ok=True)

# ── GS-SINDy import ────────────────────────────────────────────────────────────
GSINDY_PATH = os.path.join(os.getcwd(), "GS-SINDy", "GSINDy")
if os.path.isdir(GSINDY_PATH):
    sys.path.insert(0, GSINDY_PATH)
    sys.path.insert(0, os.path.join(os.getcwd(), "GS-SINDy"))
try:
    from GSINDy import GSINDy
    GSINDY_AVAILABLE = True
    print("GSINDy: ✓")
except ImportError as e:
    GSINDY_AVAILABLE = False
    print(f"GSINDy not found ({e}) — using per-trajectory PySINDy fallback.")

print(f"System : {SYSTEM.description}")
print(f"Dim    : {SYSTEM.n_dim}   Vars: {SYSTEM.var_names}")

# %% ── 1. GENERATE MULTIPLE TRAJECTORIES ──────────────────────────────────────

t_list, X_list = generate_multiple_trajectories(
    SYSTEM, X0_LIST, T_SPAN, DT, noise_level=NOISE_LEVEL, seed=SEED
)
n_traj = len(X_list)
print(f"\n{n_traj} trajectories, each shape {X_list[0].shape}")

# Phase portrait — overlay all trajectories
colors = ["royalblue", "tomato", "seagreen", "darkorange", "purple"]
if SYSTEM.n_dim == 2:
    fig, ax = plt.subplots(figsize=(6, 5))
    for i, Xi in enumerate(X_list):
        ax.plot(Xi[:, 0], Xi[:, 1], lw=0.6,
                color=colors[i % len(colors)], label=f"IC {i}")
    ax.set_xlabel(SYSTEM.var_names[0]); ax.set_ylabel(SYSTEM.var_names[1])
    ax.legend(fontsize=8)
elif SYSTEM.n_dim >= 3:
    from mpl_toolkits.mplot3d import Axes3D
    fig = plt.figure(figsize=(7, 5))
    ax  = fig.add_subplot(111, projection="3d")
    for i, Xi in enumerate(X_list):
        ax.plot(Xi[:, 0], Xi[:, 1], Xi[:, 2], lw=0.5,
                color=colors[i % len(colors)], label=f"IC {i}")
    ax.set_xlabel(SYSTEM.var_names[0])
    ax.set_ylabel(SYSTEM.var_names[1])
    ax.set_zlabel(SYSTEM.var_names[2])
    ax.legend(fontsize=7)
fig.suptitle(f"{SYSTEM_KEY} — all trajectories")
plt.tight_layout()
plt.savefig(f"results/gsindy_{SYSTEM_KEY}_phase.png", dpi=150, facecolor="none", transparent= True)
plt.show()

# %% ── 2. BUILD BASIS ─────────────────────────────────────────────────────────

# PySINDy library (used for simulation and fallback fitting)
poly_lib = ps.PolynomialLibrary(degree=POLY_DEGREE)
poly_lib.fit(X_list[0])
feature_names = poly_lib.get_feature_names(SYSTEM.var_names)
print(f"\nLibrary: {len(feature_names)} features — {feature_names}")

def feature_fn(x_arr):
    """Feature vector for simulate_from_coefficients."""
    return poly_lib.transform(np.array(x_arr).reshape(1, -1))[0]


def _build_gsindy_basis():
    """Build the basis dict expected by GSINDy from the polynomial library."""
    if CUSTOM_BASIS_FN is not None:
        return CUSTOM_BASIS_FN()

    d = SYSTEM.n_dim
    fns, nms = [], []
    for degree in range(POLY_DEGREE + 1):
        for combo in combinations_with_replacement(range(d), degree):
            pw = [0] * d
            for idx in combo:
                pw[idx] += 1
            pows = tuple(pw)
            fns.append(
                lambda *args, p=pows: np.prod([a**pi for a, pi in zip(args, p)])
            )
            term = "".join(
                "" if pi == 0 else
                (SYSTEM.var_names[i] if pi == 1
                 else f"{SYSTEM.var_names[i]}^{pi}")
                for i, pi in enumerate(pows)
            )
            nms.append(term or "1")

    arr = np.array(fns)
    return {"functions": [arr] * d,
            "names":     [np.array(nms)] * d}

# %% ── 3. FIT ──────────────────────────────────────────────────────────────────

t_ref = t_list[0]
t0    = time.time()

if GSINDY_AVAILABLE:
    basis = _build_gsindy_basis()
    gsindy = GSINDy(
        basis=basis,
        num_traj=n_traj,
        num_feature=SYSTEM.n_dim,
        threshold_sindy=THRESHOLD_SINDY,
        threshold_group=THRESHOLD_GROUP,
        threshold_similarity=THRESHOLD_SIMILARITY,
        alpha=ALPHA,
        deriv_spline=DERIV_SPLINE,
        max_iter=MAX_ITER,
        optimizer="SQTL",
        ensemble=False,
    )
    if SYSTEM.n_dim == 2:
        gsindy.get_multi_sub_series_2D(X_list, t_ref,
                                        num_series=NUM_SERIES,
                                        window_per=WINDOW_PER)
    else:
        gsindy.get_multi_sub_series_3D(X_list, t_ref,
                                        num_series=NUM_SERIES,
                                        window_per=WINDOW_PER)
    gsindy.basis_identification(remove_per=REMOVE_PER, plot_dist=False)
    Xi_final = gsindy.prediction(X_list, t_ref, split_basis=True)
    # Xi_final: (n_traj, n_dim, n_basis) — average for a consensus model
    coef          = Xi_final.mean(axis=0)
    gs_feat_names = [str(n) for n in basis["names"][0]]

else:
    # Fallback: per-trajectory PySINDy, averaged
    coef_list = []
    for Xi in X_list:
        m = ps.SINDy(
            feature_library=ps.PolynomialLibrary(degree=POLY_DEGREE),
            optimizer=ps.STLSQ(threshold=THRESHOLD_SINDY, alpha=ALPHA),
        )
        m.fit(Xi, t=DT, feature_names=SYSTEM.var_names)
        coef_list.append(m.coefficients())
    coef          = np.mean(coef_list, axis=0)
    Xi_final      = np.array(coef_list)   # (n_traj, n_dim, n_feat)
    gs_feat_names = feature_names

fit_time = time.time() - t0
print(f"\nFit time : {fit_time:.2f} s")
print_discovered_equations(coef, gs_feat_names, SYSTEM.var_names)

# %% ── 4. COEFFICIENT COMPARISON ──────────────────────────────────────────────

true_coef = build_true_coef_matrix(SYSTEM, gs_feat_names)

if true_coef is not None:
    active = np.where(
        np.any(np.abs(true_coef) > 1e-4, axis=0) |
        np.any(np.abs(coef)      > 1e-4, axis=0)
    )[0]
    dot_names = [f"{v}'" for v in SYSTEM.var_names]
    fig = plot_coefficient_comparison(
        true_coef[:, active], coef[:, active],
        [gs_feat_names[i] for i in active],
        eq_names=dot_names,
        title=f"GS-SINDy — Coefficients ({SYSTEM_KEY}, mean over trajectories)",
    )
    plt.tight_layout()
    plt.savefig(f"results/gsindy_{SYSTEM_KEY}_coefficients.png", dpi=150, facecolor="none", transparent= True)
    plt.show()
else:
    print("No ground-truth coefficients registered for this system.")

# %% ── 5. SIMULATE & COMPARE ──────────────────────────────────────────────────

eval_x0     = X0_LIST[EVAL_TRAJ]
t_test      = t_list[EVAL_TRAJ][t_list[EVAL_TRAJ] <= T_EVAL_END]
X_true_test = X_list[EVAL_TRAJ][:len(t_test)]

try:
    X_pred = simulate_from_coefficients(coef, feature_fn, eval_x0, t_test)
    sim_ok = True
except Exception as e:
    print(f"Simulation failed: {e}")
    X_pred = np.full_like(X_true_test, np.nan)
    sim_ok = False

if sim_ok:
    fig = plot_trajectories(
        t_test, X_true_test, X_pred, SYSTEM.var_names,
        title=f"GS-SINDy — Trajectory {EVAL_TRAJ} ({SYSTEM_KEY})",
    )
    plt.savefig(f"results/gsindy_{SYSTEM_KEY}_trajectory.png", dpi=150, facecolor="none", transparent= True)
    plt.show()

# %% ── 6. PER-TRAJECTORY METRICS ─────────────────────────────────────────────

print(f"\n{'Traj':>5}  " + "  ".join(f"{v:>8}" for v in SYSTEM.var_names) +
      f"  {'Total':>8}  {'Prec':>6}  {'Rec':>6}")
print("-" * (9 + 10 * SYSTEM.n_dim + 24))

for i in range(n_traj):
    t_sub = t_list[i][t_list[i] <= T_EVAL_END]
    X_sub = X_list[i][:len(t_sub)]
    c_i   = Xi_final[i] if Xi_final.ndim == 3 else coef
    try:
        Xp_i = simulate_from_coefficients(c_i, feature_fn, X0_LIST[i], t_sub)
        pd, tot = relative_l2_error(X_sub, Xp_i)
    except Exception:
        pd = [np.nan] * SYSTEM.n_dim; tot = np.nan
    pr, re = (precision_recall(true_coef.flatten(), c_i.flatten())
               if true_coef is not None else (np.nan, np.nan))
    pd_str = "  ".join(f"{v:>8.4f}" for v in pd)
    print(f"{i:>5}  {pd_str}  {tot:>8.4f}  {pr:>6.3f}  {re:>6.3f}")

per_dim, total = relative_l2_error(X_true_test, X_pred)
coef_err  = (coefficient_error(true_coef.flatten(), coef.flatten())
             if true_coef is not None else None)
prec, rec = (precision_recall(true_coef.flatten(), coef.flatten())
             if true_coef is not None else (None, None))

print_metrics("GS-SINDy (mean coef)", per_dim, total, coef_err, prec, rec,
              var_names=SYSTEM.var_names)
print(f"  Fit time : {fit_time:.3f} s")

results = dict(
    method="GS-SINDy",
    system_key=SYSTEM_KEY,
    coef=coef,
    Xi_final=Xi_final,
    true_coef=true_coef,
    feature_names=gs_feat_names,
    var_names=SYSTEM.var_names,
    **metrics_dict("GS-SINDy", per_dim, total, coef_err, prec, rec, fit_time),
    t_test=t_test,
    X_true=X_true_test,
    X_pred=X_pred,
    params=dict(threshold_sindy=THRESHOLD_SINDY, threshold_group=THRESHOLD_GROUP,
                threshold_similarity=THRESHOLD_SIMILARITY,
                noise_level=NOISE_LEVEL, poly_degree=POLY_DEGREE),
)
with open(RESULTS_FILE, "wb") as fh:
    pickle.dump(results, fh)
print(f"\nResults saved → {RESULTS_FILE}")

# %% ── 7. COEFFICIENT SPREAD ACROSS TRAJECTORIES ─────────────────────────────

if Xi_final.ndim == 3 and n_traj > 1:
    dot_names = [f"{v}'" for v in SYSTEM.var_names]
    fig, axes = plt.subplots(1, SYSTEM.n_dim,
                              figsize=(6 * SYSTEM.n_dim, 4))
    if SYSTEM.n_dim == 1:
        axes = [axes]
    for eq_i, ax in enumerate(axes):
        data = Xi_final[:, eq_i, :]
        nz   = np.where(np.any(np.abs(data) > 1e-3, axis=0))[0]
        if len(nz) == 0:
            ax.set_title(f"{dot_names[eq_i]} — all zero")
            continue
        ax.boxplot(data[:, nz], patch_artist=True,
                   labels=[str(gs_feat_names[j]) for j in nz])
        ax.axhline(0, color="k", lw=0.5)
        ax.set_title(f"{dot_names[eq_i]} — spread across {n_traj} trajectories")
        ax.set_ylabel("Coefficient value")
        plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
    plt.tight_layout()
    plt.savefig(f"results/gsindy_{SYSTEM_KEY}_coef_spread.png", dpi=150, facecolor="none", transparent= True)
    plt.show()

# %% ── 8. NOISE SWEEP ─────────────────────────────────────────────────────────

traj_errs = []

for nl in SWEEP_NOISE_LEVELS:
    _, Xl_n = generate_multiple_trajectories(
        SYSTEM, X0_LIST, T_SPAN, DT, noise_level=nl, seed=SEED
    )
    try:
        if GSINDY_AVAILABLE:
            basis = _build_gsindy_basis()
            g = GSINDy(
                basis=basis, num_traj=n_traj, num_feature=SYSTEM.n_dim,
                threshold_sindy=THRESHOLD_SINDY, threshold_group=THRESHOLD_GROUP,
                threshold_similarity=THRESHOLD_SIMILARITY,
                alpha=ALPHA, deriv_spline=DERIV_SPLINE, max_iter=MAX_ITER,
                optimizer="SQTL", ensemble=False,
            )
            if SYSTEM.n_dim == 2:
                g.get_multi_sub_series_2D(Xl_n, t_ref, num_series=NUM_SERIES,
                                           window_per=WINDOW_PER)
            else:
                g.get_multi_sub_series_3D(Xl_n, t_ref, num_series=NUM_SERIES,
                                           window_per=WINDOW_PER)
            g.basis_identification(remove_per=REMOVE_PER, plot_dist=False)
            Xi_n = g.prediction(Xl_n, t_ref, split_basis=True)
            c_nl = Xi_n.mean(axis=0)
        else:
            cl = []
            for Xi in Xl_n:
                mm = ps.SINDy(
                    feature_library=ps.PolynomialLibrary(degree=POLY_DEGREE),
                    optimizer=ps.STLSQ(threshold=THRESHOLD_SINDY, alpha=ALPHA),
                )
                mm.fit(Xi, t=DT, feature_names=SYSTEM.var_names); cl.append(mm.coefficients())
            c_nl = np.mean(cl, axis=0)
        Xp_nl = simulate_from_coefficients(c_nl, feature_fn, eval_x0, t_test)
        _, te = relative_l2_error(X_true_test, Xp_nl)
    except Exception:
        te = np.nan
    traj_errs.append(te)
    print(f"  noise={nl:.2f}  traj_err={te:.4f}")

fig = plot_noise_sweep(SWEEP_NOISE_LEVELS,
                       {"GS-SINDy": traj_errs},
                       metric="traj",
                       title=f"GS-SINDy ({SYSTEM_KEY}) — noise robustness")
plt.savefig(f"results/gsindy_{SYSTEM_KEY}_noise_sweep.png", dpi=150, facecolor="none", transparent= True)
plt.show()